## **Data Processing**
### Note, run ThimkersRemoteWork.ipynb first before running this
### You must also use the exact same kernel to keep the variables
---

In [1]:
# %pip install scikit_posthocs
# %pip install scikit-learn
# %pip install plotly

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

from scipy import stats
from scipy.stats import mannwhitneyu

from scipy.stats import pearsonr, spearmanr, levene, f_oneway, shapiro, kruskal, chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC


In [4]:

%store -r
print("Variables restored successfully.")


Variables restored successfully.


In [5]:
print("Current Variables")
print(f"target                      : {target.shape}")
print(f"current_profession_encoded  : {current_profession_encoded.shape}")
print(f"age_group                   : {age_group.shape}")
print(f"education_level             : {education_level.shape}")
print(f"employment_status           : {employment_status.shape}")
print(f"dev_type_encoded            : {dev_type_encoded.shape}")
print(f"work_years (non-null)       : {work_years.notna().sum()}")
print(f"learn_years (non-null)      : {learn_years.notna().sum()}")
print(f"org_size_ordinal            : {org_size_ordinal.shape}")
print(f"work_tool_count             : {work_tool_count.shape}")
print(f"personal_tool_count         : {personal_tool_count.shape}")
print(f"geographic_regions_encoded  : {geographic_regions_encoded.shape}")
print(f"language_features           : {language_features.shape}")
print(f"database_features           : {database_features.shape}")
print(f"platform_features           : {platform_features.shape}")
print(f"webframe_features           : {webframe_features.shape}")
print(f"devenv_features             : {devenv_features.shape}")
print(f"collab_features             : {collab_features.shape}")
print(f"aimodel_features            : {aimodel_features.shape}")
print(f"ai_industry_use             : {ai_industry_use.shape}")
print(f"ai_learn_how                : {ai_learn_how.shape}")
print(f"learncodeai_encoded         : {learncodeai_encoded.shape}")
print(f"aiselect_encoded            : {aiselect_encoded.shape}")
print(f"aiagents_encoded            : {aiagents_encoded.shape}")
print(f"aiagentchange_encoded       : {aiagentchange_encoded.shape}")
print(f"ai_technical_use            : {ai_technical_use.shape}")
print(f"ai_knowledge                : {ai_knowledge.shape}")
print(f"ai_orchestration            : {ai_orchestration.shape}")
print(f"ai_observe_secure           : {ai_observe_secure.shape}")
print(f"ai_external                 : {ai_external.shape}")

Current Variables
target                      : (49191,)
current_profession_encoded  : (49191, 4)
age_group                   : (49191, 7)
education_level             : (49191, 8)
employment_status           : (49191, 6)
dev_type_encoded            : (49191, 22)
work_years (non-null)       : 42893
learn_years (non-null)      : 43042
org_size_ordinal            : (49191,)
work_tool_count             : (49191,)
personal_tool_count         : (49191,)
geographic_regions_encoded  : (49191, 20)
language_features           : (49191, 42)
database_features           : (49191, 30)
platform_features           : (49191, 42)
webframe_features           : (49191, 28)
devenv_features             : (49191, 27)
collab_features             : (49191, 25)
aimodel_features            : (49191, 17)
ai_industry_use             : (49191, 10)
ai_learn_how                : (49191, 13)
learncodeai_encoded         : (49191, 3)
aiselect_encoded            : (49191, 5)
aiagents_encoded            : (49191, 5)
aiage

## **All Features and Train/Test Split**

In [6]:
X = pd.concat([
    current_profession_encoded,
    age_group,
    education_level,
    employment_status,
    dev_type_encoded,
    geographic_regions_encoded,
    pd.DataFrame({'org_size': org_size_ordinal}),
    pd.DataFrame({'work_exp': work_years}),
    pd.DataFrame({'years_code': learn_years}),
    pd.DataFrame({'work_tools': work_tool_count}),
    pd.DataFrame({'personal_tools': personal_tool_count}),
    language_features,
    database_features,
    platform_features,
    webframe_features,
    devenv_features,
    collab_features,
    aimodel_features,
    ai_industry_use,
    ai_learn_how,
    learncodeai_encoded,
    aiselect_encoded,
    aiagents_encoded,
    aiagentchange_encoded,
    ai_technical_use,
    ai_knowledge,
    ai_orchestration,
    ai_observe_secure,
    ai_external,
], axis=1)

y = target

# Remove NaN and set median data
X_clean = X.copy()
X_clean = X_clean.fillna(0)

# 
for col in ['work_exp', 'years_code', 'work_tools', 'personal_tools', 'org_size']:
    X_clean[col] = X_clean[col].fillna(X_clean[col].median())

print(f"Full matrix: {X_clean.shape}")
print(f"Target: {y.shape}")
print(f"Target Ratio: {y.value_counts().to_dict()}")

Full matrix: (49191, 403)
Target: (49191,)
Target Ratio: {0: 34016, 1: 15175}


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize Features
scaler  = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Train size: {X_train_scaled.shape}")
print(f"Test size: {X_test_scaled.shape}")
print(f"Train class balance: {pd.Series(y_train).value_counts().to_dict()}")
print(f"Test  class balance: {pd.Series(y_test).value_counts().to_dict()}")

Train size: (39352, 403)
Test size: (9839, 403)
Train class balance: {0: 27212, 1: 12140}
Test  class balance: {0: 6804, 1: 3035}


## **K-Nearest Neighbors (KNN)**

In [9]:
k_range = range(1, 31)
train_errors = []
test_errors  = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    train_errors.append(1 - knn.score(X_train_scaled, y_train))
    test_errors.append(1 - knn.score(X_test_scaled, y_test))
    print(f"K={k:<3}  Train Error: {train_errors[-1]:.4f}  Test Error: {test_errors[-1]:.4f}")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(k_range), y=train_errors,
    mode='lines+markers', name='Train Error',
    line=dict(color='royalblue')
))
fig.add_trace(go.Scatter(
    x=list(k_range), y=test_errors,
    mode='lines+markers', name='Test Error',
    line=dict(color='tomato')
))

# Get the best
best_k = test_errors.index(min(test_errors)) + 1
fig.add_vline(x=best_k, line_dash='dash', line_color='green',
              annotation_text=f'Best K={best_k}', annotation_position='top right')

fig.update_layout(
    title='KNN - Error Rate vs Number of Neighbors (K)',
    xaxis_title='K (n_neighbors)',
    yaxis_title='Error Rate',
    template='plotly_white',
    height=500
)
fig.show()

print(f"Best K by lowest test error : K = {best_k}")
print(f"Train error : {train_errors[best_k - 1]:.4f}")
print(f"Test error  : {test_errors[best_k - 1]:.4f}")


K=1    Train Error: 0.0004  Test Error: 0.3537
K=2    Train Error: 0.1751  Test Error: 0.3154
K=2    Train Error: 0.1751  Test Error: 0.3154
K=3    Train Error: 0.1752  Test Error: 0.3396
K=3    Train Error: 0.1752  Test Error: 0.3396
K=4    Train Error: 0.2130  Test Error: 0.3086
K=4    Train Error: 0.2130  Test Error: 0.3086
K=5    Train Error: 0.2147  Test Error: 0.3270
K=5    Train Error: 0.2147  Test Error: 0.3270
K=6    Train Error: 0.2306  Test Error: 0.3040
K=6    Train Error: 0.2306  Test Error: 0.3040
K=7    Train Error: 0.2305  Test Error: 0.3090
K=7    Train Error: 0.2305  Test Error: 0.3090
K=8    Train Error: 0.2432  Test Error: 0.3006
K=8    Train Error: 0.2432  Test Error: 0.3006
K=9    Train Error: 0.2390  Test Error: 0.3076
K=9    Train Error: 0.2390  Test Error: 0.3076
K=10   Train Error: 0.2461  Test Error: 0.3022
K=10   Train Error: 0.2461  Test Error: 0.3022
K=11   Train Error: 0.2439  Test Error: 0.3090
K=11   Train Error: 0.2439  Test Error: 0.3090
K=12   Train 

Best K by lowest test error : K = 29
Train error : 0.2640
Test error  : 0.2891


In [ ]:
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train_scaled, y_train)
knn_predictions = knn_best.predict(X_test_scaled)

cm_knn = confusion_matrix(y_test, knn_predictions)
acc_knn = accuracy_score(y_test, knn_predictions)
report_knn = classification_report(y_test, knn_predictions, target_names=['Non-Remote', 'Remote'], output_dict=True)

tn_knn, fp_knn, fn_knn, tp_knn = cm_knn.ravel()

print(f"KNN (K={best_k}) Results")
print(f"{'Metric':<25} {'Non-Remote':>12} {'Remote':>12}")
print("-" * 63)
print(f"{'Accuracy':<25} {acc_knn:>12.4f}")
print(f"{'Precision':<25} {report_knn['Non-Remote']['precision']:>12.4f} {report_knn['Remote']['precision']:>12.4f}")
print(f"{'Recall':<25} {report_knn['Non-Remote']['recall']:>12.4f} {report_knn['Remote']['recall']:>12.4f}")
print(f"{'F1 Score':<25} {report_knn['Non-Remote']['f1-score']:>12.4f} {report_knn['Remote']['f1-score']:>12.4f}")
print(f"{'Support':<25} {report_knn['Non-Remote']['support']:>12} {report_knn['Remote']['support']:>12}")
print(f"{'Macro Avg F1':<25} {report_knn['macro avg']['f1-score']:>12.4f}")
print(f"{'Weighted Avg F1':<25} {report_knn['weighted avg']['f1-score']:>12.4f}")

fig = go.Figure(data=go.Heatmap(
    z=cm_knn,
    x=['Predicted Non-Remote', 'Predicted Remote'],
    y=['Actual Non-Remote',    'Actual Remote'],
    text=[[str(tn_knn), str(fp_knn)], [str(fn_knn), str(tp_knn)]],
    texttemplate='%{text}',
    colorscale='Blues',
    showscale=True
))
fig.update_layout(
    title=f'KNN (K={best_k}) - Confusion Matrix',
    template='plotly_white',
    height=450
)
fig.show()


KNN (K=29) Results
Metric                      Non-Remote       Remote
---------------------------------------------------------------
Accuracy                        0.7109
Precision                       0.7362       0.5655
Recall                          0.9070       0.2715
F1 Score                        0.8127       0.3669
Support                         6804.0       3035.0
Macro Avg F1                    0.5898
Weighted Avg F1                 0.6752


## **Logistic Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression

C_range = [0.001, 0.01, 0.1, 1, 10, 100]
C_strings = [str(c) for c in C_range]

# Not all penalty and solver combinations are valid
hyperparam_combos = [
    ('l2', 'lbfgs', {}),
    ('l2', 'liblinear', {}),
    ('l1', 'liblinear', {}),
    ('l1', 'saga', {}),
    ('elasticnet', 'saga', {'l1_ratio': 0.5}),
    (None, 'lbfgs', {}),
]

lr_results = {}

for i, (penalty, solver, extra) in enumerate(hyperparam_combos, 1):
    label = f"{penalty or 'none'}/{solver}"
    print(f"[{i}/{len(hyperparam_combos)}] Fitting: {label}")
    train_errors, test_errors = [], []
    for C in C_range:
        model = LogisticRegression(penalty=penalty, solver=solver, C=C, max_iter=1000, random_state=42, **extra)
        model.fit(X_train_scaled, y_train)
        train_errors.append(1 - model.score(X_train_scaled, y_train))
        test_errors.append(1 - model.score(X_test_scaled, y_test))
        print(f"  C={str(C):<8}  Train Error: {train_errors[-1]:.4f}  Test Error: {test_errors[-1]:.4f}")
    lr_results[label] = {'train': train_errors, 'test': test_errors}

# Plot test error for all combos
fig = go.Figure()
for combo, errors in lr_results.items():
    fig.add_trace(go.Scatter(
        x=C_strings, y=errors['test'],
        mode='lines+markers', name=combo
    ))

fig.update_layout(
    title='Logistic Regression - Test Error vs C by Penalty/Solver',
    xaxis_title='C (Inverse Regularization Strength)',
    yaxis_title='Test Error Rate',
    template='plotly_white',
    height=500
)
fig.show()

# Find best overall combo and C
best_lr_label, best_C, best_C_idx, best_err = None, None, None, 1.0

for combo, errors in lr_results.items():
    index = errors['test'].index(min(errors['test']))
    if errors['test'][index] < best_err:
        best_err = errors['test'][index]
        best_lr_label = combo
        best_C_idx = index
        best_C = C_range[index]

best_penalty, best_solver = best_lr_label.split('/')
best_penalty = None if best_penalty == 'none' else best_penalty
best_extra = {'l1_ratio': 0.5} if best_penalty == 'elasticnet' else {}

print(f"Best combo: {best_lr_label}")
print(f"Best C: {best_C}")
print(f"Train error: {lr_results[best_lr_label]['train'][best_C_idx]:.4f}")
print(f"Test error: {best_err:.4f}")


c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_

Best combo: l1/liblinear
Best C: 0.01
Train error: 0.2633
Test error: 0.2675


In [ ]:
lr_best = LogisticRegression(penalty=best_penalty, solver=best_solver, C=best_C,
    max_iter=1000, random_state=42, **best_extra
)

lr_best.fit(X_train_scaled, y_train)
lr_predictions = lr_best.predict(X_test_scaled)

cm_lr = confusion_matrix(y_test, lr_predictions)
acc_lr = accuracy_score(y_test, lr_predictions)
report_lr = classification_report(y_test, lr_predictions, target_names=['Non-Remote', 'Remote'], output_dict=True)

tn_lr, fp_lr, fn_lr, tp_lr = cm_lr.ravel()

print(f"Logistic Regression ({best_lr_label}, C={best_C}) Results")
print(f"{'Metric':<25} {'Non-Remote':>12} {'Remote':>12}")
print("-" * 51)
print(f"{'Precision':<25} {report_lr['Non-Remote']['precision']:>12.4f} {report_lr['Remote']['precision']:>12.4f}")
print(f"{'Recall':<25} {report_lr['Non-Remote']['recall']:>12.4f} {report_lr['Remote']['recall']:>12.4f}")
print(f"{'F1 Score':<25} {report_lr['Non-Remote']['f1-score']:>12.4f} {report_lr['Remote']['f1-score']:>12.4f}")
print(f"{'Support':<25} {report_lr['Non-Remote']['support']:>12} {report_lr['Remote']['support']:>12}")
print("-" * 51)
print(f"{'Accuracy':<25} {acc_lr:>12.4f}")
print(f"{'Macro Avg F1':<25} {report_lr['macro avg']['f1-score']:>12.4f}")
print(f"{'Weighted Avg F1':<25} {report_lr['weighted avg']['f1-score']:>12.4f}")

fig = go.Figure(data=go.Heatmap(
    z=cm_lr,
    x=['Predicted Non-Remote', 'Predicted Remote'],
    y=['Actual Non-Remote',    'Actual Remote'],
    text=[[str(tn_lr), str(fp_lr)], [str(fn_lr), str(tp_lr)]],
    texttemplate='%{text}',
    colorscale='Blues',
    showscale=True
))
fig.update_layout(
    title=f'Logistic Regression ({best_lr_label}, C={best_C}) - Confusion Matrix',
    template='plotly_white',
    height=450
)
fig.show()


Logistic Regression (l1/liblinear, C=0.01) Results
Metric                      Non-Remote       Remote
---------------------------------------------------
Precision                       0.7535       0.6252
Recall                          0.9114       0.3315
F1 Score                        0.8249       0.4332
Support                         6804.0       3035.0
---------------------------------------------------
Accuracy                        0.7325
Macro Avg F1                    0.6291
Weighted Avg F1                 0.7041


In [13]:

# Getting top variables with .coef
feature_names = X_clean.columns.tolist()
coefs = lr_best.coef_[0] 

coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefs
})
coef_df['Abs'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values('Abs', ascending=False).head(20)
coef_df = coef_df.sort_values('Coefficient')

colors = ['tomato' if c < 0 else 'royalblue' for c in coef_df['Coefficient']]

fig = go.Figure(go.Bar(
    x=coef_df['Coefficient'],
    y=coef_df['Feature'],
    orientation='h',
    marker_color=colors
))
fig.update_layout(
    title=f'Logistic Regression ({best_lr_label}, C={best_C}) - Top 20 Feature Coefficients',
    xaxis_title='Coefficient Value',
    yaxis_title='Feature',
    template='plotly_white',
    height=600
)
fig.show()

print("Top 20 features by absolute coefficient:")
print(f"{'Rank':<6} {'Feature':<40} {'Coefficient':>12}")
print("-" * 60)
for rank, (index, row) in enumerate(coef_df.sort_values('Abs', ascending=False).iterrows(), 1):
    print(f"{rank:<6} {row['Feature']:<40} {row['Coefficient']:>12.4f}")


Top 20 features by absolute coefficient:
Rank   Feature                                   Coefficient
------------------------------------------------------------
1      learncodeai_others                            -0.3778
2      devtype_others                                -0.3577
3      employment_unemployed                         -0.2530
4      employment_employed                            0.2368
5      employment_others                             -0.2301
6      org_size                                       0.1810
7      region_northern_america                        0.1774
8      region_eastern_europe                          0.1467
9      work_exp                                       0.1311
10     region_south_america                           0.1161
11     employment_retired                            -0.0971
12     devtype_backend developer                      0.0890
13     devtype_student                               -0.0862
14     region_eastern_asia                  

## **Support Vector Machine (SVM)**

## **Kernel SVM**

## **Naive Bayes**

## **Random Forest**

In [14]:
n_estimators_range  = [10, 50, 100, 200, 300, 500]
n_estimators_strings = [str(n) for n in n_estimators_range]

max_depth_range = [None, 5, 10, 20]
bootstrap_range = [True, False]

rf_combos = [
    (depth, option) for depth in max_depth_range for option in bootstrap_range
]

rf_results = {}

for i, (depth, boot) in enumerate(rf_combos, 1):
    label = f"depth={'None' if depth is None else depth}/bootstrap={boot}"
    print(f"[{i}/{len(rf_combos)}] Fitting: {label}")
    rf_train_errors = []
    rf_test_errors  = []
    for n in n_estimators_range:
        rf = RandomForestClassifier(n_estimators=n, max_depth=depth, bootstrap=boot, random_state=42, 
                                    n_jobs=-1)
        rf.fit(X_train_scaled, y_train)
        rf_train_errors.append(1 - rf.score(X_train_scaled, y_train))
        rf_test_errors.append(1 - rf.score(X_test_scaled, y_test))
        print(f"n_estimators={str(n):<6}  Train Error: {rf_train_errors[-1]:.4f}  Test Error: {rf_test_errors[-1]:.4f}")
    rf_results[label] = {'train': rf_train_errors, 'test': rf_test_errors}

fig = go.Figure()
for combo, errors in rf_results.items():
    fig.add_trace(go.Scatter(
        x=n_estimators_strings, y=errors['test'],
        mode='lines+markers', name=combo
    ))

fig.update_layout(
    title='Random Forest - Test Error vs n_estimators by max_depth/bootstrap',
    xaxis_title='n_estimators',
    yaxis_title='Test Error Rate',
    template='plotly_white',
    height=500
)
fig.show()

best_rf_label, best_n, best_n_idx, best_rf_err = None, None, None, 1.0

for combo, errors in rf_results.items():
    idx = errors['test'].index(min(errors['test']))
    if errors['test'][idx] < best_rf_err:
        best_rf_err    = errors['test'][idx]
        best_rf_label  = combo
        best_n_idx     = idx
        best_n         = n_estimators_range[idx]

depth_choice, boot_choice = best_rf_label.split('/')
best_depth = None if 'None' in depth_choice else int(depth_choice.split('=')[1])
best_bootstrap = boot_choice.split('=')[1] == 'True'

print(f"Best combo: {best_rf_label}")
print(f"Best n_estimators: {best_n}")
print(f"Train error: {rf_results[best_rf_label]['train'][best_n_idx]:.4f}")
print(f"Test error: {best_rf_err:.4f}")


[1/8] Fitting: depth=None/bootstrap=True
n_estimators=10      Train Error: 0.0123  Test Error: 0.2815
n_estimators=10      Train Error: 0.0123  Test Error: 0.2815
n_estimators=50      Train Error: 0.0005  Test Error: 0.2544
n_estimators=50      Train Error: 0.0005  Test Error: 0.2544
n_estimators=100     Train Error: 0.0004  Test Error: 0.2475
n_estimators=100     Train Error: 0.0004  Test Error: 0.2475
n_estimators=200     Train Error: 0.0004  Test Error: 0.2464
n_estimators=200     Train Error: 0.0004  Test Error: 0.2464
n_estimators=300     Train Error: 0.0004  Test Error: 0.2478
n_estimators=300     Train Error: 0.0004  Test Error: 0.2478
n_estimators=500     Train Error: 0.0004  Test Error: 0.2444
[2/8] Fitting: depth=None/bootstrap=False
n_estimators=500     Train Error: 0.0004  Test Error: 0.2444
[2/8] Fitting: depth=None/bootstrap=False
n_estimators=10      Train Error: 0.0004  Test Error: 0.2729
n_estimators=10      Train Error: 0.0004  Test Error: 0.2729
n_estimators=50      

Best combo: depth=20/bootstrap=False
Best n_estimators: 300
Train error: 0.0663
Test error: 0.2432


In [ ]:
rf_best = RandomForestClassifier(n_estimators=best_n, max_depth=best_depth,
                                 bootstrap=best_bootstrap, random_state=42, n_jobs=-1)
rf_best.fit(X_train_scaled, y_train)
rf_predictions = rf_best.predict(X_test_scaled)

cm_rf = confusion_matrix(y_test, rf_predictions)
acc_rf = accuracy_score(y_test, rf_predictions)
report_rf = classification_report(y_test, rf_predictions, target_names=['Non-Remote', 'Remote'], output_dict=True)

tn_rf, fp_rf, fn_rf, tp_rf = cm_rf.ravel()

print(f"Random Forest ({best_rf_label}, n={best_n}) Results")
print(f"{'Metric':<25} {'Non-Remote':>12} {'Remote':>12}")
print("-" * 51)
print(f"{'Precision':<25} {report_rf['Non-Remote']['precision']:>12.4f} {report_rf['Remote']['precision']:>12.4f}")
print(f"{'Recall':<25} {report_rf['Non-Remote']['recall']:>12.4f} {report_rf['Remote']['recall']:>12.4f}")
print(f"{'F1 Score':<25} {report_rf['Non-Remote']['f1-score']:>12.4f} {report_rf['Remote']['f1-score']:>12.4f}")
print(f"{'Support':<25} {report_rf['Non-Remote']['support']:>12} {report_rf['Remote']['support']:>12}")
print("-" * 51)
print(f"{'Accuracy':<25} {acc_rf:>12.4f}")
print(f"{'Macro Avg F1':<25} {report_rf['macro avg']['f1-score']:>12.4f}")
print(f"{'Weighted Avg F1':<25} {report_rf['weighted avg']['f1-score']:>12.4f}")

fig = go.Figure(data=go.Heatmap(
    z=cm_rf,
    x=['Predicted Non-Remote', 'Predicted Remote'],
    y=['Actual Non-Remote',    'Actual Remote'],
    text=[[str(tn_rf), str(fp_rf)], [str(fn_rf), str(tp_rf)]],
    texttemplate='%{text}',
    colorscale='Blues',
    showscale=True
))
fig.update_layout(
    title=f'Random Forest ({best_rf_label}, n={best_n}) - Confusion Matrix',
    template='plotly_white',
    height=450
)
fig.show()


Random Forest (depth=20/bootstrap=False, n=300) Results
Metric                      Non-Remote       Remote
---------------------------------------------------
Precision                       0.7656       0.7093
Recall                          0.9345       0.3585
F1 Score                        0.8416       0.4763
Support                         6804.0       3035.0
---------------------------------------------------
Accuracy                        0.7568
Macro Avg F1                    0.6589
Weighted Avg F1                 0.7289


In [ ]:
# Feature Importance
feature_names = X_clean.columns.tolist()
importances = rf_best.feature_importances_

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
})
importance_df = importance_df.sort_values('Importance', ascending=False).head(20)
importance_df = importance_df.sort_values('Importance')

fig = go.Figure(go.Bar(
    x=importance_df['Importance'],
    y=importance_df['Feature'],
    orientation='h',
    marker_color='royalblue'
))
fig.update_layout(
    title=f'Random Forest ({best_rf_label}, n={best_n}) - Top 20 Feature Importances',
    xaxis_title='Importance Score',
    yaxis_title='Feature',
    template='plotly_white',
    height=600
)
fig.show()

print("Top 20 features by importance:")
print(f"{'Rank':<6} {'Feature':<40} {'Importance':>12}")
print("-" * 60)
for rank, (_, row) in enumerate(importance_df.sort_values('Importance', ascending=False).iterrows(), 1):
    print(f"{rank:<6} {row['Feature']:<40} {row['Importance']:>12.4f}")


Top 20 features by importance:
Rank   Feature                                    Importance
------------------------------------------------------------
1      org_size                                       0.1539
2      years_code                                     0.0471
3      work_exp                                       0.0378
4      employment_employed                            0.0357
5      devtype_others                                 0.0164
6      work_tools                                     0.0153
7      employment_independent                         0.0126
8      profession_professional dev                    0.0118
9      personal_tools                                 0.0116
10     region_northern_america                        0.0115
11     platform_amazon_web_services_aws               0.0101
12     learncodeai_others                             0.0093
13     collab_jira                                    0.0081
14     employment_unemployed                          

## **Neural Network**